In [12]:
import sys,os
import numpy as np
import pandas as pd
from tqdm import tqdm

#--fipack modules
path='/ceph24/JAM/ccocuzza/legacy/'
sys.path.insert(0,path)
os.environ['FITPACK']=path

from tools.tools    import checkdir,load,isnumeric
from tools.config   import conf,load_config,options, get_bounds
from fitlib.resman import RESMAN

# Setup JAM libraries

In [2]:
wdir = './priors'
load_config('%s/input.py'%wdir)
for process in list(conf['datasets']):
    if process not in ['idis','pidis']: del conf['datasets'][process]

conf['grid path'] = '/work/JAM/share/fitpack_grids'
resman=RESMAN(nworkers=1,parallel=False,datasets=True)


idis_thy  = resman.idis_thy
pidis_thy = resman.pidis_thy
pidis_res = resman.pidis_res

path2reps='%s/msr-inspected'%wdir

reps=os.listdir(path2reps)
def set_replica(irep):
    rep=load('%s/%s'%(path2reps,reps[irep]))    
    new_par=np.copy(resman.parman.par)
    step=list(rep['order'])[-1]
    for i in range(len(resman.parman.order)):
        for j in range(len(rep['order'][step])):
            if resman.parman.order[i]==rep['order'][step][j]:
                new_par[i]=rep['params'][step][j]
    resman.parman.set_new_params(new_par,initial=True)

IDIS HT: Using Q2 model 0 [1/Q2]
loading idis data sets 10010
loading idis data sets 10016
loading idis data sets 10020
loading idis data sets 10003
loading idis data sets 10026
loading idis data sets 10027
loading idis data sets 10028
loading idis data sets 10029
loading idis data sets 10030
loading idis data sets 10031
loading idis data sets 10032
loading idis data sets 10007
loading idis data sets 10011
loading idis data sets 10017
loading idis data sets 10021
loading idis data sets 10006
loading idis data sets 10002
loading idis data sets 10033
loading idis data sets 10041
loading idis data sets 10050
loading idis data sets 10051
SMEARING VERSION: 2.0
Loading nuclear smearing grid /ceph24/JAM/ccocuzza/legacy//grids/idis2.0/idis_10011_ng=100_mellnpts=4_dsmftype=paris_hsmftype=kpsv_tmc=AOT.npy
Loading nuclear smearing grid /ceph24/JAM/ccocuzza/legacy//grids/idis2.0/idis_10017_ng=100_mellnpts=4_dsmftype=paris_hsmftype=kpsv_tmc=AOT.npy
Loading nuclear smearing grid /ceph24/JAM/ccocuzza

# Load kinematics

In [3]:
def get_experimental_kinematics(fname):
    f = pd.read_excel(fname).to_dict(orient='list')
    return f

table_A1p=get_experimental_kinematics('example_A1p.xlsx')

# simulation

In [16]:
def gen_simulation(table):

    X  = np.array(table['X'])
    Q2 = np.array(table['Q2'])
    Elab = table['Elab'][0]
    
    M = 0.938
    Y = Q2/(2*M*X*Elab)
    tar = table['target'][0]
    obs = table['obs'][0]

    theta = None
    
    nrep = len(os.listdir(path2reps))
    npts=X.shape[0]
    THY=np.zeros((nrep,npts))
    i = 0
    for irep in tqdm(range(nrep)):
        set_replica(irep)
        g1 = pidis_thy.get_gX('g1',X,Q2,tar,process='pidis',idx=None)
        g2 = pidis_thy.get_gX('g1',X,Q2,tar,process='pidis',idx=None)
        F2 =  idis_thy.get_FX('F2',X,Q2,tar,process='pidis',idx=None)
        FL =  idis_thy.get_FX('FL',X,Q2,tar,process='pidis',idx=None)
        thy = pidis_res.get_asymmetry(X,Q2,Y,theta,obs,g1,g2,F2,FL)
        THY[irep] = thy
        #if i > 5: break
        i+=1
        
    simulation=table.copy()
    simulation['value']=np.mean(THY,axis=0)
    
    CL=0.68
    cdf_thy = np.quantile(THY,[(1-CL)/2,(1+CL)/2],axis=0)
    simulation['thy_err']=0.5*(cdf_thy[1]-cdf_thy[0])


    return simulation

In [17]:
simulation_A1p=gen_simulation(table_A1p)

100%|██████████| 410/410 [00:20<00:00, 19.92it/s]


In [18]:
print(simulation_A1p)

{'Elab': [11, 11, 11, 11, 11], 'col': ['ePIC', 'ePIC', 'ePIC', 'ePIC', 'ePIC'], 'X': [0.1, 0.2, 0.3, 0.4, 0.5], 'Q2': [5, 6, 7, 8, 9], 'target': ['p', 'p', 'p', 'p', 'p'], 'obs': ['A1', 'A1', 'A1', 'A1', 'A1'], 'stat_u%': [2, 2, 2, 2, 2], 'syst_u': [0.02, 0.02, 0.02, 0.02, 0.02], 'syst_c': [0.01, 0.01, 0.01, 0.01, 0.01], 'norm_c%': [3, 3, 3, 3, 3], 'value': array([0.18252542, 0.294329  , 0.39592056, 0.47599908, 0.52753137]), 'thy_err': array([0.00510243, 0.00562974, 0.00628057, 0.0080866 , 0.01116161])}


In [19]:
pd.DataFrame(simulation_A1p).to_excel('pseudodata/A1p.xlsx', index=False)